In [8]:
import os
print(os.getcwd())

C:\Users\AKSHITHA\Hospital_Segmentation_Study\notebooks


In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
data_path = "../dataset"

patients = pd.read_csv(f"{data_path}/patients.csv")
opd = pd.read_csv(f"{data_path}/opd_visits.csv")
admissions = pd.read_csv(f"{data_path}/admissions.csv")
lab = pd.read_csv(f"{data_path}/lab_tests.csv")
billing = pd.read_csv(f"{data_path}/billing.csv")
doctors = pd.read_csv(f"{data_path}/doctors.csv")
departments = pd.read_csv(f"{data_path}/departments.csv")
hospitals = pd.read_csv(f"{data_path}/hospitals.csv")

print("All datasets loaded successfully!")

All datasets loaded successfully!


In [3]:
opd = opd.drop_duplicates(subset="visit_id")
admissions = admissions.drop_duplicates(subset="admission_id")
lab = lab.drop_duplicates(subset="lab_test_id")
billing = billing.drop_duplicates(subset="bill_id")

print("Duplicate records removed!")

print("OPD:", opd.shape)
print("Admissions:", admissions.shape)
print("Lab Tests:", lab.shape)
print("Billing:", billing.shape)

Duplicate records removed!
OPD: (180000, 8)
Admissions: (40000, 9)
Lab Tests: (120000, 8)
Billing: (160000, 9)


In [4]:
opd["diagnosis_text"] = opd["diagnosis_text"].fillna("Unknown")

print("Missing values after treatment:")
print(opd.isnull().sum())

Missing values after treatment:
visit_id          0
patient_id        0
hospital_id       0
doctor_id         0
department_id     0
visit_date        0
diagnosis_text    0
visit_type        0
dtype: int64


In [5]:
reference_date = pd.Timestamp("2025-12-31")

def calculate_age(value):

    value = str(value)

    if "-" in value:
        dob = pd.to_datetime(value, errors="coerce")

        if pd.isna(dob):
            return np.nan

        return int((reference_date - dob).days / 365.25)

    return pd.to_numeric(value, errors="coerce")

patients["age"] = patients["age_or_dob"].apply(calculate_age)

display(patients[["age_or_dob", "age"]].head(15))

,age_or_dob,age
0,60,60
1,1996-07-04,29
2,28,28
3,1990-07-11,35
4,1991-05-21,34
5,1973-07-31,52
6,70,70
7,45,45
8,19,19
9,36,36


In [6]:
billing["corrected_amount"] = (
    billing["base_amount"] * (1 + billing["gst_rate"])
).round(2)

billing["gst_issue"] = (
    billing["amount_recorded"] >
    billing["corrected_amount"] + 0.01
)

print("GST issues detected:", billing["gst_issue"].sum())

display(billing.head(10))

GST issues detected: 12918


,bill_id,patient_id,hospital_id,doctor_id,department_id,bill_date,base_amount,gst_rate,amount_recorded,corrected_amount,gst_issue
0,BILL0000001,P029967,H001,DR023,D003,2025-06-21,2086.65,0.05,2190.98,2190.98,False
1,BILL0000002,P044033,H002,DR091,D001,2024-09-16,1276.49,0.05,1340.31,1340.31,False
2,BILL0000003,P048867,H001,DR043,D003,2025-09-14,1500.36,0.05,1650.40,1575.38,True
3,BILL0000004,P006700,H001,DR023,D003,2024-02-16,1772.28,0.05,1860.89,1860.89,False
4,BILL0000005,P008488,H003,DR124,D004,2024-10-13,1485.19,0.05,1633.71,1559.45,True
5,BILL0000006,P021842,H001,DR028,D008,2025-04-26,7062.84,0.05,7415.98,7415.98,False
6,BILL0000007,P015090,H004,DR194,D004,2024-11-06,970.84,0.18,1145.59,1145.59,False
7,BILL0000008,P016477,H003,DR107,D007,2025-11-21,2763.65,0.18,3261.11,3261.11,False
8,BILL0000009,P041304,H002,DR056,D006,2025-10-30,973.03,0.18,1148.18,1148.18,False
9,BILL0000010,P009910,H003,DR144,D004,2025-05-07,1050.30,0.05,1102.82,1102.82,False


In [7]:
clean_path = "../outputs/cleaned_data"

os.makedirs(clean_path, exist_ok=True)

patients.to_csv(f"{clean_path}/patients_cleaned.csv", index=False)
opd.to_csv(f"{clean_path}/opd_cleaned.csv", index=False)
admissions.to_csv(f"{clean_path}/admissions_cleaned.csv", index=False)
lab.to_csv(f"{clean_path}/lab_cleaned.csv", index=False)
billing.to_csv(f"{clean_path}/billing_cleaned.csv", index=False)

print("All cleaned datasets saved successfully!")

All cleaned datasets saved successfully!
